In [1]:
import numpy as np
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # 0 = all logs, 1 = filter INFO, 2 = filter WARNING, 3 = filter ERROR
os.chdir('/groups/SUDOCO/Task23/sudoco_task2.3/')
from surrogates_interface.transformers import Transformer
import matplotlib.pyplot as plt
from surrogates_interface.surrogates import TensorFlowModel
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import numpy as np
import pandas as pd
import xarray as xr
from py_wake.site import UniformSite
# from py_wake.wind_farm_models.engineering_models import PropagateDownwind
from py_wake.examples.data.hornsrev1 import V80
# from dependencies import compute_sector_average, PropagateDownwindNoSelfInduction, predict_loads_sector_average
from wind_farm_loads.py_wake import  PropagateDownwindNoSelfInduction, predict_loads_sector_average, predict_loads_rotor_average
from wind_farm_loads.tool_agnostic import compute_sector_average
from abc import ABC
from py_wake.deficit_models import ZongGaussianDeficit, NOJDeficit
from py_wake.deflection_models.jimenez import JimenezWakeDeflection
from py_wake.flow_map import HorizontalGrid
from py_wake.site._site import UniformSite
from py_wake.site.shear import PowerShear
from py_wake.turbulence_models import CrespoHernandez, GCLTurbulence,STF2017TurbulenceModel
# from py_wake.deficit_models.gaussian import GaussianDeficit
from py_wake.site import UniformSite
from py_wake.examples.data.hornsrev1 import V80, Hornsrev1Site
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Hides INFO and WARNING messages
import tensorflow as tf
tf.get_logger().setLevel('ERROR')         # Hides tf.function retracing warnings
from iea3_4_pywake_openfast_1 import iea3_4
from py_wake.wind_turbines.power_ct_functions import PowerCtTabular
from py_wake.wind_turbines import WindTurbine, WindTurbines
from data.turbine.iea_22s import IEA22s
from py_wake.deficit_models.gaussian import ZongGaussianDeficit
from py_wake.superposition_models import WeightedSum, SqrMaxSum
from py_wake.wind_farm_models import PropagateDownwind
from py_wake.turbulence_models import CrespoHernandez
from py_wake.deflection_models.jimenez import JimenezWakeDeflection
from py_wake.site._site import UniformSite
from py_wake.deficit_models.utils import ct2a_mom1d
from data.turbine.iea_22s import IEA22s
from py_wake.wind_turbines.power_ct_functions import PowerCtFunctionList, PowerCtTabular
from py_wake.rotor_avg_models import RotorCenter, GridRotorAvg, EqGridRotorAvg, GQGridRotorAvg, CGIRotorAvg, PolarGridRotorAvg, PolarRotorAvg, polar_gauss_quadrature, GaussianOverlapAvgModel
from py_wake.flow_map import HorizontalGrid


# Updated list of available surrogate channels based on files you have
channels = [
    "SA_blade_root_ip", "SA_blade_root_oop","SA_blade_root_projected", "SA_shaft_oop", "SA_shaft_yaw",
    "SA_tbss", "SA_tbfa", "SA_tower_top_fa", "SA_tower_torsion","SA_tower_top_ss","SA_tower_base_projected"
]

channels_RA = [
    "RA_blade_root_ip", "RA_blade_root_oop","RA_blade_root_projected", "RA_shaft_oop", "RA_shaft_yaw",
    "RA_tbss", "RA_tbfa", "RA_tower_top_fa", "RA_tower_torsion","RA_tower_top_ss","RA_tower_base_projected"
]

surrogates = {}
for ch in channels:
    model_path = os.path.join("models/SA_surrogate_IEA34", f"{ch}.keras")
    scaler_path = os.path.join("models/SA_surrogate_IEA34", f"scaler_{ch}.h5")
    # surrogates[ch] = TensorFlowModel(model_path, scaler_path)
    surrogates[ch] = TensorFlowModel.load_h5(
    model_path=model_path, extra_data_path=scaler_path)


surrogates_rotor = {}
for ch in channels_RA:
    model_path = os.path.join("models/RA_surrogate_IEA34", f"{ch}.keras")
    scaler_path = os.path.join("models/RA_surrogate_IEA34", f"scaler_{ch}.h5")
    # surrogates[ch] = TensorFlowModel(model_path, scaler_path)
    surrogates_rotor[ch] = TensorFlowModel.load_h5(
    model_path=model_path, extra_data_path=scaler_path)

2025-08-25 12:12:49.995344: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756116770.060711   80926 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756116770.078623   80926 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756116770.201801   80926 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756116770.201848   80926 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756116770.201850   80926 computation_placer.cc:177] computation placer alr

In [5]:
#IEA34 example

# %%prun
wt = iea3_4

u = [0,3,12,25,30]
ct = [0,0,0,0, 0]
power = [0,0,0,0,0]

wt2 = WindTurbine(name='MyWT',
                    diameter=wt.diameter(),
                    hub_height=wt.hub_height(),
                    powerCtFunction=PowerCtTabular(u,power,'kW',ct))
wts = WindTurbines.from_WindTurbine_lst([wt2, wt])

# Wind farm layout
D = wt.diameter()
x_turbines = np.array([0.0, 5, 10, 15]) * D  # [m]
y_turbines = np.array([5, 5.0, 5,5]) * D  # [m]

# models pywake
deficit_model = ZongGaussianDeficit(
    a=[0.38, 4e-3],
    deltawD=1.0 / np.sqrt(2),
    eps_coeff=0.35,
    lam=7.5,
    B=3,
    rotorAvgModel=CGIRotorAvg(21),
    groundModel=None,
    use_effective_ws=True,
    use_effective_ti=True,
)

turbulence_model = CrespoHernandez(
    ct2a=ct2a_mom1d,
    c=[0.73, 0.83, 0.03, -0.32],
    addedTurbulenceSuperpositionModel=SqrMaxSum(),
)

site = UniformSite(shear=PowerShear(h_ref=wt.hub_height(), alpha=0.2))


# Wind farm model (no self induction)
wfm = PropagateDownwindNoSelfInduction(
    site=site,
    windTurbines=wts,
    wake_deficitModel=deficit_model,
    superpositionModel=WeightedSum(),
    deflectionModel=JimenezWakeDeflection(),
    turbulenceModel=turbulence_model,
)

yaw_ang= np.array([20, 0, 20, 0])  # yaw angles for each turbine
power_lev= np.array([100, 100, 100, 100])  # power levels for each turbine
sim_res = wfm(
    x=x_turbines,
    y=y_turbines,
    type=np.full(4, True),  # Only first three turbines are active
    TI=np.full(1, 0.1),            # single turbulence intensity
    wd=np.full(1, 270),              # single wind direction
    ws=np.full(1, 8),              # single wind speed
    yaw=yaw_ang,        # array yaw values
    tilt=np.full(4, 0),      # array of tilt values
    time=True,
    power_demand=power_lev,     # array of power level values
)

# sim_res.flow_map().plot_wake_map()  # this will take a long time





# sim_res

In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
import time

# ---------------------------
# User-defined
# ---------------------------
turbine_idx = 1  # index of turbine to inspect

channels = [
    "SA_blade_root_ip", "SA_blade_root_oop", "SA_blade_root_projected",
    "SA_shaft_oop", "SA_shaft_yaw",
    "SA_tbss", "SA_tbfa", "SA_tower_top_fa",
    "SA_tower_torsion", "SA_tower_top_ss", "SA_tower_base_projected"
]

radius_values = [ 6, 7, 8, 9, 10, 11, 12, 15, 5, 6, 7, 8, 9, 10, 5, 6, 7, 8, 9, 10, 50, 100 ]
azimuth_values = [ 37, 37, 37, 37, 37, 37, 37, 37, 73, 73, 73, 73, 73, 73, 109, 109, 109, 109, 109, 109, 181, 181 ]

loads_per_channel = {ch: [] for ch in channels}
n_points = []
run_times = []
grid_strings = []

# ---------------------------
# Loop
# ---------------------------
for n_radius, n_azimuth in zip(radius_values, azimuth_values):
    print(f"Running: n_radius={n_radius}, n_azimuth={n_azimuth}, points = {n_radius*n_azimuth} ...")
    start = time.time()

    sa = compute_sector_average(
        sim_res,
        n_radius=n_radius,
        n_azimuth=n_azimuth,
        look="downwind",
    )

    if isinstance(sa, xr.Dataset):
        sa = sa.to_array(dim="quantity")

    loads = predict_loads_sector_average(
        surrogates,
        sa,
        np.full(sim_res.wt.size, 0),    # yaw
        np.full(sim_res.wt.size, 100),  # power demand
    )

    da = loads.squeeze()  # dims: ('wt','name')

    df = pd.DataFrame(
        da.values,
        columns=da.coords["name"].values,
        index=[f"T{i}" for i in range(da.sizes["wt"])]
    )

    turbine_id = f"T{turbine_idx}"
    for ch in channels:
        loads_per_channel[ch].append(df.loc[turbine_id, ch])

    elapsed = time.time() - start
    run_times.append(elapsed)
    grid_strings.append(f"{n_radius}x{n_azimuth}")
    n_points.append(n_radius * n_azimuth)

    print(f"Done in {elapsed:.2f} s")

# ---------------------------
# Compute relative differences (%)
# ---------------------------
results_df = pd.DataFrame(loads_per_channel, index=n_points)
results_df.index.name = "n_points"

baseline = results_df.iloc[-1]
diff_df = (results_df - baseline) / baseline * 100.0

# Add extra metadata columns
diff_df.insert(0, "Grid", grid_strings)
diff_df.insert(0, "Time [s]", [round(t, 2) for t in run_times])

# Add max diff column 
diff_df["Max Abs Diff (%)"] = diff_df[channels].abs().max(axis=1)

# Move index into a column
diff_df = diff_df.reset_index()

# Save to CSV with header
output_file = "sector_average_relative_diff_full.csv"
with open(output_file, "w") as f:
    f.write("=== Relative Difference (%) vs. Finest Grid ===\n")
diff_df.to_csv(output_file, mode="a", index=False)

print(f"\n Saved detailed diff table with timing to: {output_file}")

# Optional: print to console
print("\n=== Relative Difference (%) vs. Finest Grid ===")
print(diff_df.round(2))


Running: n_radius=6, n_azimuth=37, points = 222 ...
Done in 0.14 s
Running: n_radius=7, n_azimuth=37, points = 259 ...
Done in 0.13 s
Running: n_radius=8, n_azimuth=37, points = 296 ...
Done in 0.13 s
Running: n_radius=9, n_azimuth=37, points = 333 ...
Done in 0.13 s
Running: n_radius=10, n_azimuth=37, points = 370 ...
Done in 0.13 s
Running: n_radius=11, n_azimuth=37, points = 407 ...
Done in 0.14 s
Running: n_radius=12, n_azimuth=37, points = 444 ...
Done in 0.15 s
Running: n_radius=15, n_azimuth=37, points = 555 ...
Done in 0.14 s
Running: n_radius=5, n_azimuth=73, points = 365 ...
Done in 0.13 s
Running: n_radius=6, n_azimuth=73, points = 438 ...
Done in 0.14 s
Running: n_radius=7, n_azimuth=73, points = 511 ...
Done in 0.14 s
Running: n_radius=8, n_azimuth=73, points = 584 ...
Done in 0.14 s
Running: n_radius=9, n_azimuth=73, points = 657 ...
Done in 0.14 s
Running: n_radius=10, n_azimuth=73, points = 730 ...
Done in 0.14 s
Running: n_radius=5, n_azimuth=109, points = 545 ...
Done